# RAG on Dutch Wikipedia articles with OGX and pgvector

This notebook shows the simplest way to do retrieval-augmented generation (RAG) with
OGX: upload plain text files and let the OGX server chunk, embed and store them. As
data I use a dozen Dutch Wikipedia articles about the solar system and its planets.
Questions are asked in Dutch as well: all models involved are multilingual.

## What it does

1. **Pick the models.** The LLM (`qwen38-27b`) answers the questions, and an embedding
   model turns text into vectors. The notebook supports two embedding models:
   `multilingual-e5-large` (the default) and `bge-m3` (commented out).
2. **Find or create the vector store.** It reuses the pgvector store named
   `my_pgvector_store` if it exists, and only creates it otherwise, so rerunning the
   notebook doesn't leave duplicate stores behind.
3. **Fetch and upload the articles.** Each article is downloaded as plain text through
   the Wikipedia API and uploaded as a separate file. The OGX server then splits it into
   chunks, embeds them and stores them. The chunk size depends on the embedding model's
   input limit: 500 tokens (100 overlap) for e5, which accepts at most 512 tokens, and
   1500 tokens (300 overlap) for bge-m3.
4. **Ask a question twice, with a strictly grounded prompt.** The system instructions
   tell the model to answer only from the retrieved context, and otherwise to reply
   *"Information not found in provided documents."*
   - **Without retrieval**, the model has no context, so it should give exactly that
     reply, even for a well-known fact. That shows the prompt works.
   - **With `file_search`** over the vector store, it should answer from the articles.
     The output lists the search queries and the chunks that came back, so you can check
     where the answer came from.

## Good to know

- **`tool_choice="required"`** forces the model to search. Without it, the model may
  skip the search and answer from memory, despite the instructions.
- **Thinking is turned off** for the LLM
  (`extra_body={"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}}`),
  which brings answers down from minutes to seconds.
- **Compared with `docling.ipynb`:** there, Docling chunks a PDF along its structure
  before upload. Here, OGX chunks plain text by token count only, which is simpler but
  can cut through the middle of a paragraph.

In [ ]:
pip install ogx_client requests rich

In [ ]:
from ogx_client import OgxClient

client = OgxClient(base_url="http://ogx-custom-server-service.ogx.svc.cluster.local:8321")

In [ ]:
models=client.models.list()

for m in models:
    print(m)

In [ ]:
model_id = "vllm-inference/qwen38-27b"
embedding_model_id = "vllm-embedding/redhataimultilingual-e5-large"
#embedding_model_id = "vllm-embedding/baai-bge-m3"
embedding_model = next(m for m in models if m.id == embedding_model_id)
embedding_dimension = int(embedding_model.custom_metadata["embedding_dimension"])

In [ ]:
stores = client.vector_stores.list().data
store_name = "my_pgvector_store"

vector_store_id = None

for s in client.vector_stores.list().data:
    if s.name == store_name:
        vector_store_id = s.id
        break

if vector_store_id is None:
    vector_store = client.vector_stores.create(
        name=store_name,
        extra_body={
            "embedding_model": embedding_model_id,
            "embedding_dimension": embedding_dimension,
            "provider_id": "pgvector",
        },
    )
    vector_store_id = vector_store.id

In [ ]:
client.vector_stores.list()

In [ ]:
import io, time, requests

if embedding_model_id == "vllm-embedding/baai-bge-m3":
    chunk_size = 1500
    chunk_overlap = 300
elif embedding_model_id == "vllm-embedding/redhataimultilingual-e5-large":
    chunk_size = 500
    chunk_overlap = 100

print("chunk_size = " + str(chunk_size))
print("chunk_overlap = " + str(chunk_overlap))

titles = ["Zonnestelsel", "Zon", "Mercurius (planeet)", "Venus (planeet)", "Aarde (planeet)", "Maan",
          "Mars (planeet)", "Jupiter (planeet)", "Saturnus (planeet)", "Uranus (planeet)",
          "Neptunus (planeet)", "Pluto (dwergplaneet)"]

for title in titles:
    for attempt in range(5):
        r = requests.get("https://nl.wikipedia.org/w/api.php",
                         params={"action": "query", "prop": "extracts", "explaintext": 1,
                                 "redirects": 1, "format": "json", "titles": title},
                         headers={"User-Agent": "ogx-rag-demo/0.1"}, timeout=30)
        if r.status_code != 429:
            break
        time.sleep(10)  # rate-limited by Wikipedia, wait and retry
    page = next(iter(r.json()["query"]["pages"].values()))
    text = page.get("extract", "")
    if not text:
        print(f"skipped {title}: not found")
        continue

    file_info = client.files.create(
        file=(f"{page['title']}.txt", io.BytesIO(text.encode("utf-8"))),
        purpose="assistants",
    )
    vs_file = client.vector_stores.files.create(
        vector_store_id=vector_store_id,
        file_id=file_info.id,
        chunking_strategy={"type": "static",
                           "static": {"max_chunk_size_tokens": chunk_size, "chunk_overlap_tokens": chunk_overlap}},
    )
    print(f"{page['title']}: {vs_file.status} {vs_file.last_error or ''}")
    time.sleep(2)

In [ ]:
# query without a vector store
system_instructions = "You are a strictly grounded assistant. Answer the user's question using ONLY the provided [Context]. Never extrapolate, guess, or use outside world knowledge. If the context does not explicitly contain the answer, reply exactly: 'Information not found in provided documents.'"
query = "Wie ontdekte de planeet Neptunus?"


In [ ]:
# without lookup
response = client.responses.create(
    model=model_id,
    input=query,
    instructions=system_instructions,
    extra_body={"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}},
)

from IPython.display import Markdown, display

display(Markdown(response.output_text))

In [ ]:
# with lookup
response = client.responses.create(
    model=model_id,
    input=query,
    instructions=system_instructions,
    tools=[{"type": "file_search", "vector_store_ids": [vector_store_id], "max_num_results": 5}],
    include=["file_search_call.results"],# also return the retrieved chunks
    tool_choice="required", # always use tool
    extra_body={"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}},
)

for item in response.output:
    if item.type == "file_search_call":
        print(f"Search ({item.status}): {item.queries}")
        for r in item.results or []:
            #print(f"  {r.score:.3f}  {r.filename}: {r.text[:100]!r}")
            print(f"  {r.score:.3f}  {r.filename}: {r.text}")
    elif item.type == "message":
        for part in item.content:
            for a in getattr(part, "annotations", None) or []:
                if a.type == "file_citation":
                    print(f"Cited: {a.filename}")

from IPython.display import Markdown, display

display(Markdown(response.output_text))

In [ ]:
from rich import print as rprint
rprint(response)